In [5]:
"""
health_condition 予測パイプライン（本命版）
LightGBM / XGBoost / CatBoost の3モデルを5-fold CVで学習し、
OOF予測でブレンド重みを検証しつつソフト投票アンサンブルします。

実行環境: Kaggle Notebook、Google Colab、またはローカル
  pip install lightgbm xgboost catboost scikit-learn pandas numpy

入力ファイルパスは環境に合わせて INPUT_DIR を書き換えてください。
"""
import time
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OrdinalEncoder

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier



In [7]:
RANDOM_STATE = 42
N_SPLITS = 5
INPUT_DIR =  os.path.join(os.getcwd(),'src')  # train.csv / test.csv があるディレクトリに変更してください

NUM_COLS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
            'step_count', 'exercise_duration', 'water_intake']
CAT_COLS = ['diet_type', 'stress_level', 'sleep_quality',
            'physical_activity_level', 'smoking_alcohol', 'gender']


def load_data():
    train = pd.read_csv(f'{INPUT_DIR}/train.csv')
    test = pd.read_csv(f'{INPUT_DIR}/test.csv')
    return train, test


def engineer_features(df):
    df = df.copy()
    df['n_missing'] = df[NUM_COLS + CAT_COLS].isna().sum(axis=1)
    df['calorie_per_step'] = df['calorie_expenditure'] / (df['step_count'] + 1)
    df['steps_per_exercise_min'] = df['step_count'] / (df['exercise_duration'] + 1)
    df['water_per_bmi'] = df['water_intake'] / (df['bmi'] + 1e-3)
    return df


EXTRA_NUM_COLS = ['n_missing', 'calorie_per_step', 'steps_per_exercise_min', 'water_per_bmi']


def main():
    t0 = time.time()
    train, test = load_data()
    train = engineer_features(train)
    test = engineer_features(test)

    feats = NUM_COLS + EXTRA_NUM_COLS + CAT_COLS

    # ラベルエンコード（3モデルとも整数コード化したカテゴリを使う）
    le_target = {c: i for i, c in enumerate(sorted(train['health_condition'].unique()))}
    inv_target = {i: c for c, i in le_target.items()}
    y = train['health_condition'].map(le_target)

    # カテゴリ変数: 欠損は専用カテゴリとして保持
    for c in CAT_COLS:
        train[c] = train[c].astype('category')
        test[c] = test[c].astype('category')

    X = train[feats]
    X_test = test[feats]

    # CatBoost用に文字列カテゴリ（NaNは 'missing' 文字列に）
    # 注意: category dtype のまま astype(str) すると NaN が 'nan' 文字列にならず
    # 欠損のまま残ることがあるため、先に object 型に戻してから fillna する。
    X_cb = X.copy()
    X_test_cb = X_test.copy()
    for c in CAT_COLS:
        X_cb[c] = X_cb[c].astype(object).where(X_cb[c].notna(), 'missing').astype(str)
        X_test_cb[c] = X_test_cb[c].astype(object).where(X_test_cb[c].notna(), 'missing').astype(str)
        assert X_cb[c].isna().sum() == 0 and X_test_cb[c].isna().sum() == 0
    cat_idx = [X_cb.columns.get_loc(c) for c in CAT_COLS]

    # XGBoost用にOrdinalEncode（NaNはそのままにしXGBoostのmissing処理に任せる）
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_xgb = X.copy()
    X_test_xgb = X_test.copy()
    X_xgb[CAT_COLS] = oe.fit_transform(X[CAT_COLS].astype(str))
    X_test_xgb[CAT_COLS] = oe.transform(X_test[CAT_COLS].astype(str))
    # NaN復元（OrdinalEncoderはNaNも文字列'nan'にしてしまうため -1 をNaN扱いにする）
    for c in CAT_COLS:
        X_xgb.loc[X[c].isna(), c] = np.nan
        X_test_xgb.loc[X_test[c].isna(), c] = np.nan

    n_classes = len(le_target)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    oof_lgb = np.zeros((len(X), n_classes))
    oof_xgb = np.zeros((len(X), n_classes))
    oof_cb = np.zeros((len(X), n_classes))
    test_lgb = np.zeros((len(X_test), n_classes))
    test_xgb = np.zeros((len(X_test), n_classes))
    test_cb = np.zeros((len(X_test), n_classes))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f'--- Fold {fold + 1}/{N_SPLITS} ---')
        ytr, yval = y.iloc[tr_idx], y.iloc[val_idx]

        # ---------- LightGBM ----------
        lgb_train = lgb.Dataset(X.iloc[tr_idx], label=ytr, categorical_feature=CAT_COLS)
        lgb_val = lgb.Dataset(X.iloc[val_idx], label=yval, categorical_feature=CAT_COLS, reference=lgb_train)
        lgb_params = dict(
            objective='multiclass', num_class=n_classes, metric='multi_error',
            learning_rate=0.05, num_leaves=63, min_data_in_leaf=50,
            feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
            lambda_l2=1.0, verbose=-1, seed=RANDOM_STATE + fold,
        )
        model_lgb = lgb.train(lgb_params, lgb_train, num_boost_round=2000,
                               valid_sets=[lgb_val],
                               callbacks=[lgb.early_stopping(100, verbose=False)])
        oof_lgb[val_idx] = model_lgb.predict(X.iloc[val_idx])
        test_lgb += model_lgb.predict(X_test) / N_SPLITS

        # ---------- XGBoost ----------
        dtrain = xgb.DMatrix(X_xgb.iloc[tr_idx], label=ytr, enable_categorical=False)
        dval = xgb.DMatrix(X_xgb.iloc[val_idx], label=yval, enable_categorical=False)
        dtest = xgb.DMatrix(X_test_xgb, enable_categorical=False)
        xgb_params = dict(
            objective='multi:softprob', num_class=n_classes, eval_metric='merror',
            eta=0.05, max_depth=8, min_child_weight=5,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            seed=RANDOM_STATE + fold,
        )
        model_xgb = xgb.train(xgb_params, dtrain, num_boost_round=2000,
                               evals=[(dval, 'val')],
                               early_stopping_rounds=100, verbose_eval=False)
        oof_xgb[val_idx] = model_xgb.predict(dval)
        test_xgb += model_xgb.predict(dtest) / N_SPLITS

        # ---------- CatBoost ----------
        model_cb = CatBoostClassifier(
            iterations=2000, learning_rate=0.05, depth=8,
            loss_function='MultiClass', eval_metric='Accuracy',
            l2_leaf_reg=3.0, random_seed=RANDOM_STATE + fold,
            early_stopping_rounds=100, verbose=False,
        )
        model_cb.fit(X_cb.iloc[tr_idx], ytr, cat_features=cat_idx,
                     eval_set=(X_cb.iloc[val_idx], yval))
        oof_cb[val_idx] = model_cb.predict_proba(X_cb.iloc[val_idx])
        test_cb += model_cb.predict_proba(X_test_cb) / N_SPLITS

        acc_l = accuracy_score(yval, oof_lgb[val_idx].argmax(1))
        acc_x = accuracy_score(yval, oof_xgb[val_idx].argmax(1))
        acc_c = accuracy_score(yval, oof_cb[val_idx].argmax(1))
        print(f'  LightGBM acc={acc_l:.5f}  XGBoost acc={acc_x:.5f}  CatBoost acc={acc_c:.5f}')

    # ---------- ブレンド重みの簡易探索（OOFベース） ----------
    best_acc, best_w = 0, (1/3, 1/3, 1/3)
    for w1 in np.arange(0, 1.05, 0.1):
        for w2 in np.arange(0, 1.05 - w1, 0.1):
            w3 = 1 - w1 - w2
            if w3 < 0:
                continue
            blend = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cb
            acc = accuracy_score(y, blend.argmax(1))
            if acc > best_acc:
                best_acc, best_w = acc, (w1, w2, w3)

    print(f'\n最良ブレンド重み (LGB, XGB, CB) = {best_w}, OOF accuracy = {best_acc:.5f}')

    w1, w2, w3 = best_w
    test_blend = w1 * test_lgb + w2 * test_xgb + w3 * test_cb
    test_pred = np.array([inv_target[i] for i in test_blend.argmax(1)])

    submission = pd.DataFrame({'id': test['id'], 'health_condition': test_pred})
    submission.to_csv('submission_lgb_xgb_cb.csv', index=False)
    print('submission_lgb_xgb_cb.csv を保存しました')
    print(f'総実行時間: {time.time() - t0:.1f}秒')


if __name__ == '__main__':
    main()

--- Fold 1/5 ---
  LightGBM acc=0.96699  XGBoost acc=0.96704  CatBoost acc=0.96658
--- Fold 2/5 ---
  LightGBM acc=0.96770  XGBoost acc=0.96750  CatBoost acc=0.96738
--- Fold 3/5 ---
  LightGBM acc=0.96713  XGBoost acc=0.96706  CatBoost acc=0.96638
--- Fold 4/5 ---
  LightGBM acc=0.96711  XGBoost acc=0.96697  CatBoost acc=0.96664
--- Fold 5/5 ---
  LightGBM acc=0.96707  XGBoost acc=0.96692  CatBoost acc=0.96626

最良ブレンド重み (LGB, XGB, CB) = (np.float64(0.4), np.float64(0.4), np.float64(0.19999999999999996)), OOF accuracy = 0.96724
submission_lgb_xgb_cb.csv を保存しました
総実行時間: 5880.3秒
